# EventHub AI Create Event V2 — Pipeline 05: Export GGUF & Integrate Ollama
### Mục tiêu:
1. Hợp nhất trọng số LoRA Adapter vào Base Model (`merge_and_unload`).
2. Biên dịch công cụ `llama.cpp` trên máy chủ Colab.
3. Chuyển đổi mô hình 16-bit sang định dạng nhị phân **GGUF**.
4. Lượng tử hóa mô hình sang **Q4_K_M** (chuẩn cân bằng hoàn hảo giữa tốc độ và độ chính xác trích xuất JSON).
5. Tạo `Modelfile` tùy biến cho Ollama và chạy kiểm tra khói (Smoke Test) trước khi đưa về Backend EventHub.



In [ ]:
# [1] Cài đặt dependencies và Mount Drive
!pip install -q -U torch transformers peft accelerate

from google.colab import drive
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/EventHub_AI'
ADAPTER_DIR = os.path.join(DRIVE_ROOT, 'adapters/eventhub_qwen_v2')
MERGED_DIR = os.path.join(DRIVE_ROOT, 'merged_model/eventhub-extractor-v2')
GGUF_DIR = os.path.join(DRIVE_ROOT, 'export/gguf')

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)



In [ ]:
# [2] Hợp nhất (Merge) LoRA Adapter vào Base Model
import json

# Tự động nhận diện chính xác Base Model từ file cấu hình của Adapter
adapter_cfg_file = os.path.join(ADAPTER_DIR, 'adapter_config.json')
if os.path.exists(adapter_cfg_file):
    with open(adapter_cfg_file, 'r', encoding='utf-8') as f:
        cfg = json.load(f)
    BASE_MODEL_ID = cfg.get("base_model_name_or_path", "Qwen/Qwen3-4B")
else:
    BASE_MODEL_ID = "Qwen/Qwen3-4B"

print(f"Đang tải Base Model chính xác: {BASE_MODEL_ID} ở định dạng FP16...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu", # Merge trên RAM để tránh tràn VRAM
    trust_remote_code=True
)

print(f"Đang gắn LoRA Adapter từ {ADAPTER_DIR}...")
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

print("Đang hợp nhất trọng số (merge_and_unload)...")
merged_model = peft_model.merge_and_unload()

print(f"Đang lưu mô hình hợp nhất vào: {MERGED_DIR}")
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("✅ Hợp nhất mô hình hoàn tất!")



In [ ]:
# [3] Clone và Build llama.cpp trên Colab
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!pip install -r requirements.txt
!cmake -B build
!cmake --build build --config Release -j
print("Biên dịch llama.cpp hoàn tất!")



In [ ]:
# [4] Chuyển đổi mô hình HF sang GGUF (FP16)
gguf_f16_path = os.path.join(GGUF_DIR, 'eventhub-extractor-v2-f16.gguf')
!python convert_hf_to_gguf.py {MERGED_DIR} --outfile {gguf_f16_path} --outtype f16
print(f"Đã xuất GGUF FP16: {gguf_f16_path}")



In [ ]:
# [5] Lượng tử hóa sang định dạng Q4_K_M
gguf_q4_path = os.path.join(GGUF_DIR, 'eventhub-extractor-v2-Q4_K_M.gguf')
!./build/bin/llama-quantize {gguf_f16_path} {gguf_q4_path} Q4_K_M
print(f"✅ Xuất thành công GGUF Q4_K_M: {gguf_q4_path}")



In [ ]:
# [6] Tạo tệp Modelfile cho Ollama
modelfile_content = f"""FROM ./eventhub-extractor-v2-Q4_K_M.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
"""

SYSTEM """Bạn là chuyên gia trích xuất thông tin sự kiện của hệ thống EventHub. Chỉ trả về duy nhất chuỗi JSON hợp lệ tuân thủ schema, không thêm giải thích hoặc markdown."""

PARAMETER temperature 0.1
PARAMETER top_p 0.95
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
"""

modelfile_path = os.path.join(GGUF_DIR, 'Modelfile')
with open(modelfile_path, 'w', encoding='utf-8') as f:
    f.write(modelfile_content)

print(f"Đã tạo Modelfile tại: {modelfile_path}")



## Hướng dẫn nhập mô hình vào Ollama trên máy Backend:
1. Tải 2 tệp về máy local:
   - `eventhub-extractor-v2-Q4_K_M.gguf`
   - `Modelfile`
2. Mở Terminal / PowerShell tại thư mục chứa 2 file trên và chạy lệnh:
   ```bash
   ollama create eventhub-extractor-v2 -f Modelfile
   ```
3. Kiểm tra mô hình trong danh sách:
   ```bash
   ollama list
   ```
4. Cập nhật biến môi trường trong file `.env` của Backend EventHub:
   ```env
   AI_MODEL_NAME=eventhub-extractor-v2
   ```
   *(Hệ thống sẽ chuyển sang dùng model mới mà không cần sửa code business logic!)*

